# 🪷 CHÙA LINH ỨNG ONLINE — TIKTOK LIVE INTERACTIVE 8K
### HỆ THỐNG PHÁT SÓNG TƯƠNG TÁC TÂM LINH THỜI GIAN THỰC TRÊN GOOGLE COLAB

Hệ thống phát sóng tương tác chuẩn khung hình dọc **9:16 (1080x1920)** với đồ họa Chánh Điện Đức Phật A Di Đà dát vàng 8K sắc nét, hỗ trợ 2 hình thức phát sóng:
- **Cách 1 (Khuyên dùng - Nhẹ máy nhất):** Colab làm máy chủ -> Cấp link HTTPS Cloudflare Tunnel để dán vào mục **Liên kết** trong **TikTok LIVE Studio** hoặc **OBS Studio** (Không bao giờ bị lỗi màn hình đen hay bắt đăng nhập).
- **Cách 2 (Treo máy tự động 24/7):** Colab tự tạo Màn hình ảo (Xvfb 1080x1920) và dùng FFmpeg đẩy luồng trực tiếp lên TikTok bằng **Stream Key** mà không cần bật máy tính cá nhân!


In [ ]:
# @title 🚀 BƯỚC 1: Khởi Chạy Server Chùa Linh Ứng 8K & Lấy Link Dán Vào TikTok LIVE Studio { display-mode: "form" }
# @markdown Bấm nút **Play** bên trái để khởi động toàn bộ hệ thống:

TikTok_Username = "@2166541467" # @param {type:"string"}
Bat_Giong_Doc_AI = True # @param {type:"boolean"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/4] Đang đồng bộ mã nguồn Chùa Online 8K...")
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/app 2>/dev/null || (cd /content/app && git pull)
%cd /content/app

print("⏳ [2/4] Đang kiểm tra thư viện Node.js & Cloudflare Tunnel...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Cài đặt ZeroTTS nếu chọn bật giọng đọc AI
if Bat_Giong_Doc_AI:
    try:
        import zerotts
    except ImportError:
        print("🎙️ Đang cài đặt thư viện giọng đọc AI ZeroTTS tiếng Việt...")
        !pip install --quiet zerotts pillow

print("⏳ [3/4] Đang dọn dẹp tiến trình cũ và khởi chạy Server...")
!pkill -9 -f "server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared" > /dev/null 2>&1
time.sleep(1)

# Khởi chạy server.js ở chế độ nền
with open("/tmp/server.log", "w") as f_log:
    server_proc = subprocess.Popen(["node", "server.js"], cwd="/content/app", stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

# Khởi chạy Cloudflare Tunnel công khai
print("⏳ [4/4] Đang khởi tạo đường truyền HTTPS công khai...")
with open("/tmp/cloudflared.log", "w") as cf_log:
    tunnel_proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3000"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(15):
    time.sleep(1)
    if os.path.exists("/tmp/cloudflared.log"):
        with open("/tmp/cloudflared.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    obs_link = f"{public_url}/?obs=1"
    admin_link = f"{public_url}/admin"
    html_card = f"""
    <div style="background: linear-gradient(135deg, #2b0805 0%, #150302 100%); border: 2px solid #e5a93c; border-radius: 16px; padding: 22px; color: #fffaf0; font-family: sans-serif; box-shadow: 0 8px 30px rgba(229,169,60,0.3);">
        <h2 style="color: #ffd700; margin-top: 0; font-size: 22px;">🎉 CHÙA LINH ỨNG ONLINE 8K ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #f5deb3;">
            Đường truyền công khai đã được thiết lập qua Cloudflare Tunnel (<b>Không bị hỏi đăng nhập GitHub, không bị lỗi màn hình đen</b>).
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{obs_link}" target="_blank" style="background: linear-gradient(180deg, #d97706 0%, #b45309 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #ffd700; box-shadow: 0 4px 15px rgba(217,119,6,0.4);">
                📺 [1] MỞ MÀN HÌNH LIVE (Dán link này vào TikTok LIVE Studio)
            </a>
            <a href="{admin_link}" target="_blank" style="background: linear-gradient(180deg, #15803d 0%, #166534 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #86efac;">
                🎛️ [2] MỞ TRANG QUẢN TRỊ & THỬ NGHIỆM NÚT BẤM
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.4); border-left: 4px solid #ffd700; padding: 10px 14px; border-radius: 6px; font-size: 13.5px; color: #ffe4b5;">
            📌 <b>Link dán vào TikTok LIVE Studio (Nguồn Liên Kết):</b><br>
            <code style="color: #fff; background: #000; padding: 4px 8px; border-radius: 4px; display: inline-block; margin-top: 5px; font-size: 14px;">{obs_link}</code>
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Không lấy được link Cloudflare. Kiểm tra log:")
    !cat /tmp/cloudflared.log


In [ ]:
# @title ⚔️ BƯỚC 1B: Khởi Chạy Game ĐẠI CHIẾN ĐỎ VS XANH 9:16 { display-mode: "form" }
# @markdown Bấm nút Play bên trái để khởi chạy Game Đấu Đội Đỏ vs Xanh:

TikTok_Username = "@2166541467" # @param {type:"string"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/3] Đang cập nhật mã nguồn Game Đỏ vs Xanh...")
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/app 2>/dev/null || (cd /content/app && git pull)
%cd /content/app/games/battle-red-vs-blue

print("⏳ [2/3] Cài đặt dependencies...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Dọn dẹp tiến trình cũ
!pkill -9 -f "games/battle-red-vs-blue/server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared" > /dev/null 2>&1
time.sleep(1)

# Khởi chạy game
print("⏳ [3/3] Đang khởi chạy Server Game Đỏ vs Xanh...")
env = os.environ.copy()
env["PORT"] = "3001"
env["TIKTOK_USERNAME"] = TikTok_Username.replace("@", "").strip()

with open("/tmp/battle_server.log", "w") as f_log:
    subprocess.Popen(["node", "server.js"], cwd="/content/app/games/battle-red-vs-blue", env=env, stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

with open("/tmp/battle_cf.log", "w") as cf_log:
    subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3001"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(15):
    time.sleep(1)
    if os.path.exists("/tmp/battle_cf.log"):
        with open("/tmp/battle_cf.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    html_card = f"""
    <div style="background: linear-gradient(135deg, #3d0718 0%, #0c0214 100%); border: 2px solid #ff3b8d; border-radius: 16px; padding: 22px; color: #fffaf0; font-family: sans-serif; box-shadow: 0 8px 30px rgba(255,59,141,0.3);">
        <h2 style="color: #00e5d4; margin-top: 0; font-size: 22px;">⚔️ GAME ĐẠI CHIẾN ĐỎ VS XANH ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #ffebf3;">
            Trò chơi kéo co / phá tháp 2 phe thời gian thực đã được kích hoạt thành công qua Cloudflare Tunnel.
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{public_url}" target="_blank" style="background: linear-gradient(180deg, #ff3b8d 0%, #c41865 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #ff7eb3; box-shadow: 0 4px 15px rgba(255,59,141,0.4);">
                🎮 [1] MỞ MÀN HÌNH GAME 9:16 (Dán vào TikTok LIVE Studio)
            </a>
            <a href="{public_url}/admin" target="_blank" style="background: linear-gradient(180deg, #00b4d8 0%, #0077b6 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #90e0ef;">
                🎛️ [2] TRANG ADMIN TEST NÚT BẤM (Không cần mật khẩu)
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.5); border-left: 4px solid #00e5d4; padding: 10px 14px; border-radius: 6px; font-size: 13.5px; color: #d0f4de;">
            📌 <b>Link dán vào TikTok LIVE Studio (Nguồn Liên Kết):</b><br>
            <code style="color: #fff; background: #000; padding: 4px 8px; border-radius: 4px; display: inline-block; margin-top: 5px; font-size: 14px;">{public_url}</code>
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Không lấy được link Cloudflare. Kiểm tra log:")
    !cat /tmp/battle_cf.log


In [ ]:
# @title 👑 BƯỚC 1C: Khởi Chạy Game VUA TIẾNG VIỆT (ĐẢO CHỮ ĐOÁN TỪ) 9:16 { display-mode: "form" }
# @markdown Bấm nút **Play** bên trái để khởi chạy Game Đoán Từ Tương Tác Vua Tiếng Việt:

TikTok_Username = "@2166541467" # @param {type:"string"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/3] Đang cập nhật mã nguồn Game Vua Tiếng Việt...")
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/app 2>/dev/null || (cd /content/app && git pull)
%cd /content/app/games/vua-tieng-viet

print("⏳ [2/3] Cài đặt dependencies...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Dọn dẹp tiến trình cũ
!pkill -9 -f "games/vua-tieng-viet/server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared.*3002" > /dev/null 2>&1
time.sleep(1)

# Khởi chạy server Vua Tiếng Việt trên port 3002
print("⏳ [3/3] Đang khởi chạy Server Game Vua Tiếng Việt...")
env = os.environ.copy()
env["PORT"] = "3002"
env["TIKTOK_USERNAME"] = TikTok_Username.strip()

with open("/tmp/vt_server.log", "w") as f_log:
    subprocess.Popen(["node", "server.js"], cwd="/content/app/games/vua-tieng-viet", env=env, stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

with open("/tmp/vt_cf.log", "w") as cf_log:
    subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3002"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(15):
    time.sleep(1)
    if os.path.exists("/tmp/vt_cf.log"):
        with open("/tmp/vt_cf.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    html_card = f"""
    <div style="background: radial-gradient(circle at 50% 20%, #1e133d 0%, #0d0822 65%, #05030f 100%); border: 2px solid #ffd700; border-radius: 16px; padding: 22px; color: #fffaf0; font-family: sans-serif; box-shadow: 0 8px 30px rgba(255,215,0,0.3);">
        <h2 style="color: #ffd700; margin-top: 0; font-size: 22px;">👑 GAME VUA TIẾNG VIỆT ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #ddd6fe;">
            Game đảo chữ đoán từ tương tác trực tiếp qua bình luận và quà tặng TikTok Live đã khởi động thành công.
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{public_url}" target="_blank" style="background: linear-gradient(180deg, #f59e0b 0%, #d97706 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #ffd700; box-shadow: 0 4px 15px rgba(245,158,11,0.4);">
                📺 [1] MỞ MÀN HÌNH GAME 9:16 (Dán vào TikTok LIVE Studio)
            </a>
            <a href="{public_url}/admin.html" target="_blank" style="background: linear-gradient(180deg, #8b5cf6 0%, #6d28d9 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #c4b5fd;">
                🎛️ [2] BẢNG ĐIỀU KHIỂN STREAMER (Đổi câu, mở chữ, cộng giờ)
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.5); border-left: 4px solid #ffd700; padding: 10px 14px; border-radius: 6px; font-size: 13.5px; color: #fde047;">
            📌 <b>Link dán vào TikTok LIVE Studio (Nguồn Liên Kết):</b><br>
            <code style="color: #fff; background: #000; padding: 4px 8px; border-radius: 4px; display: inline-block; margin-top: 5px; font-size: 14px;">{public_url}</code>
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Không lấy được link Cloudflare. Kiểm tra log:")
    !cat /tmp/vt_cf.log


In [ ]:
# @title 🏇 BƯỚC 1D: Khởi Chạy Game ĐUA TỈNH THÀNH (HORSE & PROVINCE RACE) 9:16 { display-mode: "form" }
# @markdown Bấm nút **Play** bên trái để khởi chạy Game Đua Ngựa / Đua Tỉnh Thành Kích Nạp Quà & Tăng Tương Tác:

TikTok_Username = "@2166541467" # @param {type:"string"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/3] Đang cập nhật mã nguồn Game Đua Tỉnh Thành...")
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/app 2>/dev/null || (cd /content/app && git pull)
%cd /content/app/games/dua-tinh-thanh

print("⏳ [2/3] Cài đặt dependencies...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Dọn dẹp tiến trình cũ
!pkill -9 -f "games/dua-tinh-thanh/server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared.*3003" > /dev/null 2>&1
time.sleep(1)

# Khởi chạy server Đua Tỉnh Thành trên port 3003
print("⏳ [3/3] Đang khởi chạy Server Game Đua Tỉnh Thành...")
env = os.environ.copy()
env["PORT"] = "3003"
env["TIKTOK_USERNAME"] = TikTok_Username.strip()

with open("/tmp/dtt_server.log", "w") as f_log:
    subprocess.Popen(["node", "server.js"], cwd="/content/app/games/dua-tinh-thanh", env=env, stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

with open("/tmp/dtt_cf.log", "w") as cf_log:
    subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3003"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(15):
    time.sleep(1)
    if os.path.exists("/tmp/dtt_cf.log"):
        with open("/tmp/dtt_cf.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    html_card = f"""
    <div style="background: radial-gradient(circle at 50% 15%, #1b1035 0%, #0c081e 60%, #05030e 100%); border: 2px solid #38bdf8; border-radius: 16px; padding: 22px; color: #fffaf0; font-family: sans-serif; box-shadow: 0 8px 30px rgba(56,189,248,0.3);">
        <h2 style="color: #38bdf8; margin-top: 0; font-size: 22px;">🏇 GAME ĐUA TỈNH THÀNH ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #e2e8f0;">
            Game đua ngựa 5 tỉnh thành (Hà Nội, Sài Gòn, Đà Nẵng, Hải Phòng, Cần Thơ) tương tác thời gian thực qua Quà tặng (Hoa Hồng, Kem, Tim, Bánh Rán, TikTok) và Bình luận đã khởi động thành công.
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{public_url}" target="_blank" style="background: linear-gradient(180deg, #0284c7 0%, #0369a1 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #38bdf8; box-shadow: 0 4px 15px rgba(2,132,199,0.4);">
                📺 [1] MỞ MÀN HÌNH GAME 9:16 (Dán vào TikTok LIVE Studio)
            </a>
            <a href="{public_url}/admin.html" target="_blank" style="background: linear-gradient(180deg, #ec4899 0%, #be185d 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #f472b6;">
                🎛️ [2] BẢNG ĐIỀU KHIỂN STREAMER (Test quà, xuất phát, reset)
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.5); border-left: 4px solid #38bdf8; padding: 10px 14px; border-radius: 6px; font-size: 13.5px; color: #7dd3fc;">
            📌 <b>Link dán vào TikTok LIVE Studio (Nguồn Liên Kết):</b><br>
            <code style="color: #fff; background: #000; padding: 4px 8px; border-radius: 4px; display: inline-block; margin-top: 5px; font-size: 14px;">{public_url}</code>
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Không lấy được link Cloudflare. Kiểm tra log:")
    !cat /tmp/dtt_cf.log


In [ ]:
# @title 🏮 BƯỚC 1E: Khởi Chạy Game THẢ ĐÈN HOA ĐĂNG CẦU NGUYỆN 9:16 { display-mode: "form" }
# @markdown Bấm nút **Play** bên trái để khởi chạy Game Thả Đèn Hoa Đăng (Không lo dính bản quyền âm thanh & chính sách tâm linh):

TikTok_Username = "@2166541467" # @param {type:"string"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/3] Đang cập nhật mã nguồn Game Thả Đèn Hoa Đăng...")
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/app 2>/dev/null || (cd /content/app && git pull)
%cd /content/app/games/tha-den-hoa-dang

print("⏳ [2/3] Cài đặt dependencies...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Dọn dẹp tiến trình cũ
!pkill -9 -f "games/tha-den-hoa-dang/server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared.*3004" > /dev/null 2>&1
time.sleep(1)

# Khởi chạy server Thả Đèn Hoa Đăng trên port 3004
print("⏳ [3/3] Đang khởi chạy Server Game Thả Đèn Hoa Đăng...")
env = os.environ.copy()
env["PORT"] = "3004"
env["TIKTOK_USERNAME"] = TikTok_Username.strip()

with open("/tmp/thaden_server.log", "w") as f_log:
    subprocess.Popen(["node", "server.js"], cwd="/content/app/games/tha-den-hoa-dang", env=env, stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

with open("/tmp/thaden_cf.log", "w") as cf_log:
    subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3004"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(15):
    time.sleep(1)
    if os.path.exists("/tmp/thaden_cf.log"):
        with open("/tmp/thaden_cf.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    html_card = f"""
    <div style="background: radial-gradient(circle at 50% 20%, #171133 0%, #08061a 60%, #03020a 100%); border: 2px solid #fbbf24; border-radius: 16px; padding: 22px; color: #fffaf0; font-family: sans-serif; box-shadow: 0 8px 30px rgba(251,191,36,0.3);">
        <h2 style="color: #fbbf24; margin-top: 0; font-size: 22px;">🏮 GAME THẢ ĐÈN HOA ĐĂNG ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #e2e8f0;">
            Dòng sông ánh trăng lung linh huyền ảo, khán giả bình luận để thả đèn hoa đăng mang lời ước, tặng quà để thả Đèn Sen Vàng & Đèn Trời Khổng Minh (Âm thanh suối chảy & chuông gió sạch 100%).
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{public_url}" target="_blank" style="background: linear-gradient(180deg, #d97706 0%, #b45309 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #fbbf24; box-shadow: 0 4px 15px rgba(217,119,6,0.4);">
                📺 [1] MỞ MÀN HÌNH GAME 9:16 (Dán vào TikTok LIVE Studio)
            </a>
            <a href="{public_url}/admin.html" target="_blank" style="background: linear-gradient(180deg, #7c3aed 0%, #6d28d9 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #a78bfa;">
                🎛️ [2] BẢNG ĐIỀU KHIỂN STREAMER (Test lời ước, quà tặng, dọn sông)
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.5); border-left: 4px solid #fbbf24; padding: 10px 14px; border-radius: 6px; font-size: 13.5px; color: #fde047;">
            📌 <b>Link dán vào TikTok LIVE Studio (Nguồn Liên Kết):</b><br>
            <code style="color: #fff; background: #000; padding: 4px 8px; border-radius: 4px; display: inline-block; margin-top: 5px; font-size: 14px;">{public_url}</code>
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Không lấy được link Cloudflare. Kiểm tra log:")
    !cat /tmp/thaden_cf.log


In [ ]:
# @title 🍵 BƯỚC 1F: Khởi Chạy Game TRÀ ĐÁ VỈA HÈ HÀ NỘI 9:16 { display-mode: "form" }
# @markdown Bấm nút **Play** bên trái để khởi chạy Game Trà Đá Vỉa Hè (Bốc đầu xe, chôm xe, ghế VIP 1 & 2):

TikTok_Username = "@2166541467" # @param {type:"string"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/3] Đang cập nhật mã nguồn Game Trà Đá Vỉa Hè...")
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/app 2>/dev/null || (cd /content/app && git pull)
%cd /content/app/games/tra-da-via-he

print("⏳ [2/3] Cài đặt dependencies...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Dọn dẹp tiến trình cũ
!pkill -9 -f "games/tra-da-via-he/server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared.*3005" > /dev/null 2>&1
time.sleep(1)

# Khởi chạy server Trà Đá Vỉa Hè trên port 3005
print("⏳ [3/3] Đang khởi chạy Server Game Trà Đá Vỉa Hè Hà Nội...")
env = os.environ.copy()
env["PORT"] = "3005"
env["TIKTOK_USERNAME"] = TikTok_Username.strip()

with open("/tmp/trada_server.log", "w") as f_log:
    subprocess.Popen(["node", "server.js"], cwd="/content/app/games/tra-da-via-he", env=env, stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

with open("/tmp/trada_cf.log", "w") as cf_log:
    subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3005"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(15):
    time.sleep(1)
    if os.path.exists("/tmp/trada_cf.log"):
        with open("/tmp/trada_cf.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    html_card = f"""
    <div style="background: radial-gradient(circle at 50% 20%, #1c1917 0%, #0c0a09 60%, #000 100%); border: 2px solid #eab308; border-radius: 16px; padding: 22px; color: #fffaf0; font-family: sans-serif; box-shadow: 0 8px 30px rgba(234,179,8,0.3);">
        <h2 style="color: #eab308; margin-top: 0; font-size: 22px;">🍵 GAME TRÀ ĐÁ VỈA HÈ HÀ NỘI ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #e2e8f0;">
            Quán trà đá đường phố Hà Nội: Thả 100 tim vào ghế VIP 1 (3 ghế), Follow vào ghế VIP 2 (4 ghế), người sau đẩy người trước! Lệnh bốc đầu xe, chôm xe, gọi công an, ném chai, đoán chữ cực vui nhộn kèm giọng nói AI (TTS).
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{public_url}" target="_blank" style="background: linear-gradient(180deg, #ca8a04 0%, #a16207 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #facc15; box-shadow: 0 4px 15px rgba(202,138,4,0.4);">
                📺 [1] MỞ MÀN HÌNH GAME 9:16 (Dán vào TikTok LIVE Studio)
            </a>
            <a href="{public_url}/admin.html" target="_blank" style="background: linear-gradient(180deg, #2563eb 0%, #1d4ed8 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #60a5fa;">
                🎛️ [2] BẢNG ĐIỀU KHIỂN & TEST GIẢ LẬP (Test ghế VIP, bốc đầu, chôm xe, quà tặng)
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.5); border-left: 4px solid #eab308; padding: 10px 14px; border-radius: 6px; font-size: 13.5px; color: #fef08a;">
            📌 <b>Link dán vào TikTok LIVE Studio (Nguồn Liên Kết):</b><br>
            <code style="color: #fff; background: #000; padding: 4px 8px; border-radius: 4px; display: inline-block; margin-top: 5px; font-size: 14px;">{public_url}</code>
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Không lấy được link Cloudflare. Kiểm tra log:")
    !cat /tmp/trada_cf.log


In [ ]:
# @title 🎓 BƯỚC 1G: Khởi Chạy Game LỚP HỌC THÔNG MINH (BẠN CÓ THÔNG MINH HƠN HỌC SINH LỚP 5?) 9:16 { display-mode: "form" }
# @markdown Bấm nút **Play** bên trái để khởi chạy Game Lớp Học Thông Minh (Bảng trắc nghiệm, 40 bàn học, Lớp trưởng #1, Follow #2, Giọng đọc AI TTS):

TikTok_Username = "@2166541467" # @param {type:"string"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/3] Đang cập nhật mã nguồn Game Lớp Học Thông Minh...")
!git clone https://github.com/tuna1710/TikTok-Live-Interactive-Game.git /content/app 2>/dev/null || (cd /content/app && git pull)
%cd /content/app/games/lop-hoc-thong-minh

print("⏳ [2/3] Cài đặt dependencies...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Dọn dẹp tiến trình cũ
!pkill -9 -f "games/lop-hoc-thong-minh/server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared.*3006" > /dev/null 2>&1
time.sleep(1)

# Khởi chạy server Lớp Học Thông Minh trên port 3006
print("⏳ [3/3] Đang khởi chạy Server Game Lớp Học Thông Minh (Port 3006)...")
env = os.environ.copy()
env["PORT"] = "3006"
env["TIKTOK_USERNAME"] = TikTok_Username.strip()

with open("/tmp/lophoc_server.log", "w") as f_log:
    subprocess.Popen(["node", "server.js"], cwd="/content/app/games/lop-hoc-thong-minh", env=env, stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

with open("/tmp/lophoc_cf.log", "w") as cf_log:
    subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3006"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(15):
    time.sleep(1)
    if os.path.exists("/tmp/lophoc_cf.log"):
        with open("/tmp/lophoc_cf.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    html_card = f"""
    <div style="background: radial-gradient(circle at 50% 20%, #064e3b 0%, #022c22 60%, #000 100%); border: 2px solid #34d399; border-radius: 16px; padding: 22px; color: #fffaf0; font-family: sans-serif; box-shadow: 0 8px 30px rgba(52,211,153,0.35);">
        <h2 style="color: #34d399; margin-top: 0; font-size: 22px;">🎓 GAME LỚP HỌC THÔNG MINH ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #e2e8f0;">
            Phòng học tiểu học chuẩn dọc 9:16: Bảng đen trắc nghiệm, 40 bàn học sinh (nhận avatar & bảng tên khi comment), bục Lớp Trưởng quyền lực khi gửi hoa hồng/quà tặng, tích hợp giọng nói Cô giáo AI (TTS) ưu tiên đọc: <b>1: Lớp trưởng, 2: Follow, 3: Đọc câu hỏi</b>.
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{public_url}" target="_blank" style="background: linear-gradient(180deg, #059669 0%, #047857 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #6ee7b7; box-shadow: 0 4px 15px rgba(5,150,105,0.45);">
                📺 [1] MỞ MÀN HÌNH GAME 9:16 (Dán vào TikTok LIVE Studio)
            </a>
            <a href="{public_url}/admin.html" target="_blank" style="background: linear-gradient(180deg, #2563eb 0%, #1d4ed8 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #60a5fa;">
                🎛️ [2] BẢNG ĐIỀU KHIỂN & TEST GIẢ LẬP (Test Lớp Trưởng #1, Follow #2, Đáp Án)
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.5); border-left: 4px solid #34d399; padding: 10px 14px; border-radius: 6px; font-size: 13.5px; color: #a7f3d0;">
            📌 <b>Link dán vào TikTok LIVE Studio (Nguồn Liên Kết):</b><br>
            <code style="color: #fff; background: #000; padding: 4px 8px; border-radius: 4px; display: inline-block; margin-top: 5px; font-size: 14px;">{public_url}</code>
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Không lấy được link Cloudflare. Kiểm tra log:")
    !cat /tmp/lophoc_cf.log


### 📺 HƯỚNG DẪN CÀI ĐẶT VÀO TIKTOK LIVE STUDIO (CHỈ MẤT 30 GIÂY)
1. Mở phần mềm **TikTok LIVE Studio** trên máy tính của bạn.
2. Bấm vào nút **"Thêm nguồn"** $\rightarrow$ chọn mục **"Liên kết"** (biểu tượng hình quả cầu 🌐).
3. Dán đường link `https://xxxx.trycloudflare.com/?obs=1` (được sinh ra ở Bước 1 ở trên).
4. Đặt độ phân giải là: **`1080 x 1920`** (chuẩn khung dọc 9:16 của TikTok).
5. Tích chọn **"Bật âm thanh"** để khán giả nghe được tiếng mõ tụng kinh và đại hồng chung.
6. Bấm **Bắt đầu phát LIVE**, sau đó mở link Quản trị `[2]` để kết nối tài khoản TikTok của bạn!


---
## 🔴 PHẦN 2: TỰ ĐỘNG PHÁT LIVESTREAM LÊN TIKTOK TỪ COLAB (KHÔNG CẦN BẬT MÁY TÍNH)
Phần này sẽ khởi tạo **màn hình ảo 1080x1920** ngay bên trong Google Colab, mở Chánh Điện Chùa Linh Ứng và dùng **FFmpeg** đẩy thẳng luồng video 60FPS + âm thanh chuông mõ lên kênh TikTok của bạn bằng **Stream Key**!


In [ ]:
# @title 🎥 BƯỚC 2: Nhập Stream Key & Khởi Chạy Livestream Tự Động 24/7 { display-mode: "form" }
# @markdown Chọn Game muốn phát và nhập Server URL & Stream Key từ TikTok LIVE Studio / RTMP:

Chon_Game_Phat = "Lớp Học Thông Minh (Port 3006)" # @param ["Lớp Học Thông Minh (Port 3006)", "Trà Đá Vỉa Hè Hà Nội (Port 3005)", "Thả Đèn Hoa Đăng (Port 3004)", "Đua Tỉnh Thành (Port 3003)", "Vua Tiếng Việt (Port 3002)", "Đại Chiến Đỏ Xanh (Port 3001)", "Chùa Linh Ứng (Port 3000)"]
TikTok_Server_URL = "rtmp://live-push.tiktok.com/live/" # @param {type:"string"}
TikTok_Stream_Key = "" # @param {type:"string"}

import os, time, subprocess

port_map = {
    "Lớp Học Thông Minh (Port 3006)": "3006",
    "Trà Đá Vỉa Hè Hà Nội (Port 3005)": "3005",
    "Thả Đèn Hoa Đăng (Port 3004)": "3004",
    "Đua Tỉnh Thành (Port 3003)": "3003",
    "Vua Tiếng Việt (Port 3002)": "3002",
    "Đại Chiến Đỏ Xanh (Port 3001)": "3001",
    "Chùa Linh Ứng (Port 3000)": "3000"
}
game_port = port_map.get(Chon_Game_Phat, "3005")

if not TikTok_Stream_Key.strip():
    print("⚠️ VUI LÒNG NHẬP STREAM KEY CỦA TIKTOK VÀO Ô TRÊN TRƯỚC KHI BẤM CHẠY!")
else:
    print("⏳ Đang cài đặt trình duyệt headless và màn hình ảo Xvfb 1080x1920...")
    !apt-get update -qq > /dev/null 2>&1
    !apt-get install -y -qq xvfb pulseaudio ffmpeg > /dev/null 2>&1
    
    # Dọn dẹp tiến trình phát cũ
    !pkill -9 -f Xvfb > /dev/null 2>&1
    !pkill -9 -f chrome > /dev/null 2>&1
    !pkill -9 -f ffmpeg > /dev/null 2>&1
    time.sleep(1)
    
    # Khởi tạo màn hình ảo 1080x1920 24-bit
    print("📺 Đang khởi tạo Màn hình ảo dọc 1080x1920...")
    subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1080x1920x24"]) 
    os.environ["DISPLAY"] = ":99"
    time.sleep(2)
    
    # Mở trình duyệt Chrome toàn màn hình vào link game
    game_target_url = f"http://localhost:{game_port}/index.html?obs=1"
    print(f"🌐 Đang mở Game ({Chon_Game_Phat}) trong màn hình ảo: {game_target_url}...")
    chrome_cmd = [
        "google-chrome",
        "--no-sandbox",
        "--disable-dev-shm-usage",
        "--window-size=1080,1920",
        "--start-fullscreen",
        "--kiosk",
        "--autoplay-policy=no-user-gesture-required",
        game_target_url
    ]
    subprocess.Popen(chrome_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    
    # Bắt đầu đẩy luồng RTMP lên TikTok
    rtmp_destination = TikTok_Server_URL.rstrip("/") + "/" + TikTok_Stream_Key.lstrip("/")
    print(f"📡 ĐANG PHÁT TRỰC TIẾP LÊN TIKTOK TẠI: {TikTok_Server_URL}...")
    
    # Tự động nhận diện GPU NVIDIA (T4 trên Colab) để dùng phần cứng h264_nvenc
    has_gpu = (os.system("nvidia-smi > /dev/null 2>&1") == 0 and os.system("ffmpeg -encoders 2>/dev/null | grep -q h264_nvenc") == 0)
    if has_gpu:
        print("🚀 PHÁT HIỆN GPU T4: Tự động kích hoạt bộ tăng tốc phần cứng NVENC (h264_nvenc)!")
        vcodec_opts = "-c:v h264_nvenc -preset p4 -tune ll -b:v 3500k -maxrate 4000k -bufsize 7000k"
    else:
        print("ℹ️ Đang chạy trên CPU: Kích hoạt bộ mã hóa libx264 veryfast...")
        vcodec_opts = "-c:v libx264 -preset veryfast -b:v 3000k -maxrate 3500k -bufsize 6000k"
    
    ffmpeg_cmd = (
        f"ffmpeg -y -f x11grab -draw_mouse 0 -video_size 1080x1920 -framerate 30 -i :99.0 "
        f"-f lavfi -i anullsrc=channel_layout=stereo:sample_rate=44100 "
        f"{vcodec_opts} -pix_fmt yuv420p -g 60 "
        f"-c:a aac -b:a 128k -ar 44100 "
        f"-f flv "{rtmp_destination}" > /tmp/ffmpeg_stream.log 2>&1 &"
    )
    os.system(ffmpeg_cmd)
    time.sleep(3)
    print("🎉 LIVESTREAM TỰ ĐỘNG ĐÃ KHỞI CHẠY THÀNH CÔNG! Kiểm tra trạng thái ở Bước 3 bên dưới.")


In [ ]:
# @title 📊 BƯỚC 3: Kiểm Tra Tốc Độ & Trạng Thái Luồng Phát TikTok
!tail -n 25 /tmp/ffmpeg_stream.log 2>/dev/null || echo "Chưa có log phát sóng."


In [ ]:
# @title ⏹️ BƯỚC 4: Dừng Toàn Bộ Hệ Thống Livestream
!pkill -9 -f ffmpeg
!pkill -9 -f chrome
!pkill -9 -f Xvfb
!pkill -9 -f "server.js"
!pkill -9 -f cloudflared
print("🛑 ĐÃ DỪNG TOÀN BỘ PHIÊN LIVESTREAM AN TOÀN!")
